# 14 — 多行为异构图与商品推荐

上一课中所有实体共享一个编号空间，R-GCN 用整数 `edge_type` 区分关系。本课使用 PyTorch Geometric 的 `HeteroData`：用户、商品、类目、品牌拥有独立的节点空间；点击、加购、购买等关系拥有独立的边张量。最终任务是根据历史图为用户排序尚未购买的商品。

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from crosscity.data.heterogeneous_recommendation import make_toy_ecommerce_graph
from crosscity.models.heterogeneous_recommendation import (
    HeterogeneousEmbedding, HeterogeneousGraphSAGE,
)
from crosscity.training.heterogeneous_recommendation import train_heterogeneous_recommender
from crosscity.utils import seed_everything
if torch.cuda.is_available(): device = torch.device('cuda')
elif torch.backends.mps.is_available(): device = torch.device('mps')
else: device = torch.device('cpu')
device

## 1. 同构图和异构图有什么区别？

同构图只有一种节点和一种边。LightGCN 虽然包含用户和电影，但把它们偏移后放进同一个节点矩阵，所有交互边使用同一传播规则。

异构图至少存在多种节点类型或边类型：

```text
用户 ─点击→ 商品 ─属于→ 类目
用户 ─加购→ 商品 ─来自→ 品牌
用户 ─购买→ 商品
```

`点击` 表示弱兴趣，`加购` 更接近购买意图，`购买` 是目标行为；把它们全部压成同一种边会丢失语义。

In [ ]:
full_data = make_toy_ecommerce_graph()
graph = full_data.graph
print('node types:', graph.node_types)
print('edge types:')
for edge_type in graph.edge_types:
    print(' ', edge_type, tuple(graph[edge_type].edge_index.shape))
print('metadata =', graph.metadata())

## 2. `HeteroData` 如何保存数据？

节点通过字符串类型索引：`graph['user']`、`graph['product']`。商品 0 和用户 0 不冲突，因为它们位于两个独立空间。边类型是三元组：

$$e=(\text{source type},\text{relation},\text{target type})$$

例如 `('user','clicks','product')` 的 `edge_index` 形状为 `[2,E_click]`，第一行是局部 user ID，第二行是局部 product ID。这与知识图谱的 `(head, relation, tail)` 语义一致，但存储方式按类型拆开，更适合不同类型拥有不同特征维度的情况。

In [ ]:
clicks = graph['user', 'clicks', 'product'].edge_index
print(clicks)
print('user IDs:', clicks[0].tolist())
print('product IDs:', clicks[1].tolist())
print('num users/products:', graph['user'].num_nodes, graph['product'].num_nodes)

## 3. 为什么必须添加反向边？

消息传递沿 `source → target` 方向发生。`user ─clicks→ product` 让商品聚合点击它的用户，却不能让用户聚合自己点击过的商品。因此额外加入：

```text
product ─rev_clicks→ user
```

它不表示现实中出现了一种新行为，只是为神经网络建立反向信息通道。正向和反向使用不同参数，因为“用户给商品的信息”和“商品给用户的信息”含义不同。验证和测试购买边不会进入传播图，否则模型会提前看到答案。

In [ ]:
forward = graph['user', 'purchases', 'product'].edge_index
reverse = graph['product', 'rev_purchases', 'user'].edge_index
print('forward first edges:
', forward[:, :4])
print('reverse first edges:
', reverse[:, :4])
assert torch.equal(reverse, forward.flip(0))
held_out = set(zip(full_data.validation_users.tolist(), full_data.validation_products.tolist()))
assert held_out.isdisjoint(set(map(tuple, forward.t().tolist())))

## 4. 每种节点先拥有自己的 embedding

微型数据没有文本、图片或价格特征，所以先为每种节点建立 ID embedding：

$$E_{user}\in\mathbb R^{N_u\times d},\quad E_{product}\in\mathbb R^{N_p\times d}$$
$$E_{category}\in\mathbb R^{N_c\times d},\quad E_{brand}\in\mathbb R^{N_b\times d}$$

无传播基线只训练这些表，并用 $s(u,p)=e_u^T e_p$ 排序商品。它对应上一课中的知识图谱 embedding 基线：先确认传播真的有增益，再赞美 GNN。真实系统还能把商品标题、图片、价格编码后投影到同一隐藏维度。

In [ ]:
num_nodes = {node_type: graph[node_type].num_nodes for node_type in graph.node_types}
baseline = HeterogeneousEmbedding(num_nodes, hidden_channels=16)
representations = baseline.encode(graph)
{node_type: tuple(value.shape) for node_type, value in representations.items()}

## 5. 异构 GraphSAGE 的两级聚合

先在一种关系内部聚合。对关系 $r=(s,r,t)$，目标节点 $i$ 接收源类型邻居：

$$m_{i,r}=W_{r,self}h_i+W_{r,neighbor}\cdot mean_{j\in\mathcal N_r(i)}h_j$$

`clicks`、`adds_to_cart`、`purchases` 各有自己的 $W_r$，因此点击邻居不会被迫使用购买邻居的变换。然后将所有到达同一目标类型的关系结果聚合：

$$h_i'=h_i+\sum_{r:\,target(r)=type(i)}ReLU(m_{i,r})$$

第一层商品可以收到用户行为、类目和品牌信息；借助反向边，用户可以收到商品信息。第二层让用户间接获得商品所属类目以及相似商品的信号。残差 $+h_i$ 避免完全覆盖自身 embedding。

In [ ]:
gnn = HeterogeneousGraphSAGE(
    num_nodes, graph.metadata(), hidden_channels=16, num_layers=2
)
updated = gnn.encode(graph)
print({node_type: tuple(value.shape) for node_type, value in updated.items()})
print('number of relation-specific convolutions:', len(gnn.layers[0].convs))

## 6. 为什么最终仍然使用 BPR？

异构 GNN 负责生成用户和商品表示，解码器仍用内积：

$$s(u,p)=h_u^T h_p$$

对用户的已购买商品 $p^+$ 和未观察商品 $p^-$：

$$\mathcal L_{BPR}=-\log\sigma(s(u,p^+)-s(u,p^-))$$

这说明“图编码器”和“推荐训练目标”是两个可组合部件。换成异构图不代表 Recall/NDCG、负采样和防泄漏原则失效。未购买仍不一定等于不喜欢，所以负样本仍可能包含潜在正例。

## 7. 受控实验设计

比较三组：

1. `Embedding`：完全不传播。
2. `Purchase-only GNN`：只保留购买关系，检验普通协同传播。
3. `Full heterogeneous GNN`：加入点击、加购、类目和品牌。

三组使用同一购买训练/验证/测试目标。若第 3 组胜过第 2 组，收益才能归因于辅助关系；若 GNN 胜过 embedding，才能说明传播本身有帮助。微型数据只有 6 个用户，因此必须看多个 seed，不能把结果称为真实电商 SOTA。

In [ ]:
purchase_only = make_toy_ecommerce_graph(include_behaviors=False, include_metadata=False)
experiments = {
    'Embedding': (full_data, lambda d: HeterogeneousEmbedding(
        {t: d.graph[t].num_nodes for t in d.graph.node_types}, 16)),
    'Purchase-only GNN': (purchase_only, lambda d: HeterogeneousGraphSAGE(
        {t: d.graph[t].num_nodes for t in d.graph.node_types}, d.graph.metadata(), 16, 2)),
    'Full heterogeneous GNN': (full_data, lambda d: HeterogeneousGraphSAGE(
        {t: d.graph[t].num_nodes for t in d.graph.node_types}, d.graph.metadata(), 16, 2)),
}
rows, histories = [], {}
for seed in (42, 43, 44):
    for name, (experiment_data, build) in experiments.items():
        seed_everything(seed)
        result = train_heterogeneous_recommender(
            build(experiment_data), experiment_data, device=device, seed=seed,
            max_epochs=300, patience=60, k=3,
        )
        rows.append({
            'seed': seed, 'model': name, 'best_epoch': result.best_epoch,
            'validation_recall@3': result.validation.recall,
            'validation_ndcg@3': result.validation.ndcg,
            'test_recall@3': result.test.recall, 'test_ndcg@3': result.test.ndcg,
        })
        histories[(seed, name)] = result.history
results = pd.DataFrame(rows)
display(results)
results.groupby('model')[['validation_recall@3', 'validation_ndcg@3', 'test_recall@3', 'test_ndcg@3']].agg(['mean', 'std'])

In [ ]:
for (seed, name), history in histories.items():
    if seed == 42:
        frame = pd.DataFrame(history)
        plt.plot(frame.epoch, frame.validation_ndcg, label=name)
plt.xlabel('epoch'); plt.ylabel('validation NDCG@3')
plt.title('Does typed auxiliary information help purchase ranking?')
plt.legend(); plt.show()

## 8. 如何解释结果，而不是只看冠军

- `Purchase-only GNN < Embedding`：购买图传播像上一课 LightGCN 一样引入噪声。
- `Full GNN > Purchase-only GNN`：行为或元数据关系提供额外信号。
- `Full GNN > Embedding`：在当前划分下，关系感知传播的总收益超过噪声。
- `Full GNN < Embedding`：可能是辅助关系无效、模型过深、样本太小，或点击边与未来购买之间没有稳定联系；不能据此断言异构 GNN 普遍无效。

真实电商还必须有时间戳：只能使用推荐发生前的点击和加购。若把购买后的点击放进训练图，即使没有直接放入测试购买边，也属于时间泄漏。

## 本课结论

1. 异构图同时区分节点类型和边类型。
2. PyG 用 node dictionary 和 edge-type dictionary 保存不同张量。
3. 反向边是消息通道，不是新的现实行为。
4. 异构 GraphSAGE 先在关系内聚合，再聚合到达同一节点类型的多种关系。
5. GNN 编码器和 BPR 推荐目标是可分离的两个模块。
6. 辅助关系是否有用必须通过 purchase-only/full 的受控实验判断。
7. 下一步可以学习 HGT：用注意力自动决定不同邻居与关系的重要性，而不是简单求和。